# ROCProfiler-Compute Jupyter Analysis Example

This notebook demonstrates how to use the Jupyter interface for rocprofiler-compute analysis.

## Prerequisites

1. You need to have profiling data collected using `rocprof-compute profile`
2. The data should be in a directory (e.g., `/path/to/workload_dir`)

## Basic Usage

In [ ]:
# Import the Jupyter interface
import sys
from pathlib import Path

# Add the src directory to Python path
src_path = Path('../src').resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import rocprof_compute_jupyter as rc

## Load Performance Data

Use `rc.open()` to load your profiling data:

In [ ]:
# Replace with your actual data directory
perf_data_dir = '/path/to/your/workload_dir'

# Load the data
rc.open(perf_data_dir)

## Display Analysis Results

Use `rc.analysis()` to display the analysis results:

In [ ]:
# Display all basic metrics
rc.analysis()

## Advanced Usage

### Filter by Kernel

You can filter results to show specific kernels:

In [ ]:
# Show analysis for specific kernel IDs
rc.analysis(filter_kernel=['0', '1'])

### Filter by GPU

Filter results by GPU ID:

In [ ]:
# Show analysis for specific GPU
rc.analysis(filter_gpu=[0])

### Filter by Dispatch

Filter results by dispatch ID:

In [ ]:
# Show analysis for specific dispatches
rc.analysis(filter_dispatch=[0, 1, 2])

## Working with DataFrames

### List Available Tables

See what tables are available:

In [ ]:
# List all available table IDs
rc.list_tables()

### Get Specific DataFrames

You can retrieve specific tables as pandas DataFrames for custom analysis:

In [ ]:
# Get kernel top stats (table ID 1)
kernel_stats = rc.get_dataframe(1)
print(kernel_stats.head())

In [ ]:
# Get system info (table ID 101)
sys_info = rc.get_dataframe(101)
print(sys_info)

### Custom Analysis

Once you have the DataFrames, you can perform custom analysis:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Example: Plot kernel execution times
kernel_stats = rc.get_dataframe(1)
if kernel_stats is not None and 'Mean_Duration' in kernel_stats.columns:
    plt.figure(figsize=(12, 6))
    kernel_stats.plot(x='Kernel_Name', y='Mean_Duration', kind='bar')
    plt.title('Kernel Execution Times')
    plt.xlabel('Kernel')
    plt.ylabel('Mean Duration (ns)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Loading Data with Options

You can specify various options when loading data:

In [ ]:
# Load with custom options
rc.open(
    perf_data_dir,
    time_unit='us',           # Use microseconds instead of nanoseconds
    normal_unit='per_cycle',  # Normalize by cycles
    max_stat_num=20,          # Show top 20 statistics
    decimal=3                 # Use 3 decimal places
)

## Complete Example Workflow

In [ ]:
# 1. Load data
rc.open('/path/to/workload_dir')

# 2. Show basic overview
rc.analysis(show_basic_only=True)

# 3. List available tables
rc.list_tables()

# 4. Get specific kernel data
kernel_df = rc.get_dataframe(1)

# 5. Filter and analyze specific kernels
if kernel_df is not None:
    # Get kernel IDs for the top 3 kernels by duration
    top_kernels = kernel_df.nlargest(3, 'Mean_Duration').index.tolist()
    top_kernel_ids = [str(k) for k in top_kernels]
    
    # Show detailed analysis for these kernels
    rc.analysis(filter_kernel=top_kernel_ids)

## Notes

- The interface automatically detects your GPU architecture (MI100, MI200, MI300, etc.)
- Roofline analysis is displayed automatically if roofline data is available
- All Plotly charts are interactive - you can zoom, pan, and hover for details
- Use `show_basic_only=True` to see only high-level metrics without filters
- Apply filters to see detailed low-level metrics for specific kernels/GPUs/dispatches